# Chatbot desk vs human desk: tickets from raw weeks

**Two-Sample Test for Means (Table Format)** · a worked example from the StatNex teaching guide

**The story.** A company runs two customer-support desks: one where a chatbot answers first and hands hard cases to staff, and one staffed only by humans. It recorded tickets handled per week, 9 weeks for the chatbot desk and 7 weeks for the human desk.

**The question.** Does the chatbot desk handle more tickets per week than the human desk?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `03_two_sample/two_sample_mean_table.csv` (9 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Chatbot desk,Human desk
1.0,1604.0,1204.0
2.0,1567.0,1280.0
3.0,1460.0,1012.0
4.0,1596.0,993.0
5.0,1636.0,1292.0
6.0,1421.0,1341.0
7.0,1401.0,1168.0
8.0,1276.0,
9.0,1566.0,

These cells save `cohen_d_interval.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile cohen_d_interval.py
"""Cohen's d with its noncentral-t interval, and Hedges' exact small-sample correction.

SciPy has the noncentral t distribution (stats.nct) but no function that inverts it for
an interval, so the inversion is written out (Steiger & Fouladi 1997). The interval is
for δ = t * scale: 1/sqrt(n) for one sample or paired differences, sqrt(1/n1 + 1/n2) for
two groups with the pooled standard deviation.
"""

import math

import numpy as np
from scipy import optimize, special, stats


def _limit(t, df, q):
    """The noncentrality under which the observed t sits at the q quantile of t(df, ncp).

    The cdf at t falls as the noncentrality rises, so the bracket opens outwards from t,
    doubling, until it holds the root.
    """
    def gap(ncp):
        return stats.nct.cdf(t, df, ncp) - q

    step = 1.0
    while gap(t - step) <= 0:
        step *= 2
    low = t - step
    step = 1.0
    while gap(t + step) >= 0:
        step *= 2
    return optimize.brentq(gap, low, t + step, xtol=1e-14, rtol=4 * np.finfo(float).eps,
                           maxiter=500)


def cohen_d_interval(d, df, scale, level):
    """(lower, upper): the level interval for d = t * scale."""
    t = d / scale
    return _limit(t, df, (1 + level) / 2) * scale, _limit(t, df, (1 - level) / 2) * scale


def hedges_j(df):
    """J = Γ(ν/2) / (sqrt(ν/2) Γ((ν - 1)/2)), Hedges (1981); g = J * d."""
    return math.exp(special.gammaln(df / 2) - special.gammaln((df - 1) / 2)) / math.sqrt(df / 2)

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
groups = ["Chatbot desk", "Human desk"]
confidence = 95 / 100
alpha = 5 / 100

## Descriptive statistics

In [ ]:
# One column per group. Text and blank cells are skipped, so the groups may differ
# in size.
values = [pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float) for name in groups]
descriptive = pd.DataFrame({"n": [len(v) for v in values], "mean": [v.mean() for v in values],
                            "variance": [v.var(ddof=1) for v in values]}, index=groups)
descriptive["SD"] = np.sqrt(descriptive["variance"])
descriptive["SE"] = descriptive["SD"] / np.sqrt(descriptive["n"])
print(descriptive)

## Test for equal variances (F test, two-sided)

In [ ]:
# The larger variance goes on top. The F ratio assumes normal data and is very
# sensitive to heavy tails; stats.levene is the robust check. Read it as a
# description, not as a switch: choosing the pooled or the Welch t test from this
# result distorts the P value of the t test, so when in doubt report Welch's.
(n1, n2), (v1, v2) = descriptive["n"], descriptive["variance"]
if v2 > v1:
    f, df1, df2 = v2 / v1, n2 - 1, n1 - 1
else:
    f, df1, df2 = v1 / v2, n1 - 1, n2 - 1
p_f = min(1.0, 2 * min(stats.f.sf(f, df1, df2), stats.f.cdf(f, df1, df2)))
homogeneity = pd.DataFrame([[f, df1, df2, p_f]], columns=["F", "df1", "df2", "P"])
print()
print(homogeneity.to_string(index=False))

## Ranks

In [ ]:
# Every test below is repeated on the ranks of the pooled observations (ties share
# their mean rank): a robust alternative that keeps the t-test framework.
ranks = stats.rankdata(np.concatenate(values))
rank_values = [ranks[:len(values[0])], ranks[len(values[0]):]]
ranked = pd.DataFrame({"n": [len(v) for v in rank_values],
                       "mean": [v.mean() for v in rank_values],
                       "variance": [v.var(ddof=1) for v in rank_values]}, index=groups)
samples = {"observed": descriptive, "ranked": ranked}

## Effect sizes, confidence intervals, t tests and power

In [ ]:
# Both t tests are written from (n, mean, variance) so they work on summary data
# too, which stats.ttest_ind cannot take; on raw data stats.ttest_ind(x, y,
# equal_var=True / False) gives the same t. The pooled ("Student") test assumes equal
# variances; Welch's does not, and loses little when they are equal. Both assume
# independent observations from roughly normal populations.
mean_difference = abs(descriptive["mean"].iloc[0] - descriptive["mean"].iloc[1])
effect, intervals, tests = {}, [], []
for label, d in samples.items():
    (n1, n2), (m1, m2), (v1, v2) = d["n"], d["mean"], d["variance"]
    difference = m1 - m2
    pooled_ss = (n1 - 1) * v1 + (n2 - 1) * v2
    # Cohen's d (SD on N) divides by the pooled SD over N, Cohen's d_s by the pooled SD
    # over N - 2; neither applies the small-sample correction, which Hedges' g_s adds.
    effect[label] = [abs(difference) / np.sqrt(pooled_ss / (n1 + n2)),
                     abs(difference) / np.sqrt(pooled_ss / (n1 + n2 - 2))]
    se_pooled = np.sqrt(pooled_ss / (n1 + n2 - 2)) * np.sqrt(1 / n1 + 1 / n2)
    se_welch = np.sqrt(v1 / n1 + v2 / n2)
    df_welch = (v1 / n1 + v2 / n2) ** 2 / ((v1 / n1) ** 2 / (n1 - 1) + (v2 / n2) ** 2 / (n2 - 1))
    for method, se, df in (("t test", se_pooled, n1 + n2 - 2), ("Welch", se_welch, df_welch)):
        t = difference / se
        p = 2 * stats.t.sf(abs(t), df)
        # Power: the chance a two-sided test at alpha rejects if the true effect equals
        # the one observed (non-central t with ncp = |t|). Computed from the observed effect,
        # it adds nothing to the P value; plan power from an effect that matters.
        critical = stats.t.ppf(1 - alpha / 2, df)
        power = stats.nct.sf(critical, df, abs(t)) + stats.nct.cdf(-critical, df, abs(t))
        tests.append([label, method, abs(t), df, p, stars(p), power])
        if label == "observed":
            # The interval is for |mean 1 - mean 2| and is always two-sided.
            half = stats.t.ppf(0.5 + confidence / 2, df) * se
            intervals.append([method, abs(difference) - half, abs(difference) + half])

effect = pd.DataFrame.from_dict(effect, orient="index", columns=["Cohen's d (SD on N)", "Cohen's d_s"])
intervals = pd.DataFrame(intervals, columns=["method", "lower", "upper"])
tests = pd.DataFrame(tests, columns=["data", "method", "t", "df", "P", "", "power"])
print(f"\nDifference of means |mean 1 - mean 2| = {mean_difference:.6g}")
print(effect)
print(f"\n{confidence:.0%} confidence interval for the difference")
print(intervals.to_string(index=False))
print()
print(tests.to_string(index=False))

## Effect sizes, with confidence intervals

In [ ]:
# Cohen's d_s = (mean 1 - mean 2) / pooled SD over N - 2: the Cohen's d_s above with its
# sign, and Student's t times sqrt(1/n1 + 1/n2). Its interval inverts the noncentral t
# at that t (Steiger & Fouladi 1997); Hedges' g_s = J * d_s removes d_s's small-sample
# bias, and its interval is d_s's times J. Both steps are in cohen_d_interval.py.
from cohen_d_interval import cohen_d_interval, hedges_j


def size(value):
    """Cohen's (1988) reading of |d|: 0.2 small, 0.5 medium, 0.8 large."""
    return ["negligible", "small", "medium", "large"][sum(abs(value) >= s for s in (0.2, 0.5, 0.8))]


(n1, n2), (m1, m2), (v1, v2) = descriptive["n"], descriptive["mean"], descriptive["variance"]
df = n1 + n2 - 2
d_s = (m1 - m2) / np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / df)
lower, upper = cohen_d_interval(d_s, df, np.sqrt(1 / n1 + 1 / n2), confidence)
j = hedges_j(df)
effect_sizes = pd.DataFrame(
    [[d_s, lower, upper, size(d_s)], [j * d_s, j * lower, j * upper, size(j * d_s)]],
    index=["Cohen's d_s", "Hedges' g_s"], columns=["value", "lower", "upper", "size"],
)
print()
print(effect_sizes)

## Normality: the Shapiro-Wilk test on each group

In [ ]:
normality = pd.DataFrame([[len(v), *stats.shapiro(v)] for v in values], index=groups,
                         columns=["n", "W", "P"])
print()
print(normality)

## What the result says

The chatbot desk handled 1,503 tickets a week on average, the human desk 1,184. The difference of means is about 319 tickets.

The t statistic is 4.96 and the P value is 0.0002. The P value is the chance of a gap at least this large if both desks really handled the same number. The 95% confidence interval, the range of values the data agree with for the true gap, runs from 181 to 456 tickets a week. Cohen's d, the gap divided by the standard deviation, is 2.5, a large effect. The checks agree: the test of homogeneity of variance (P = 0.71) shows no difference in spread, and Welch's method and the tests on ranks give the same verdict.

So the chatbot desk handles more tickets. But a ticket count measures volume, not whether customers' problems were solved.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.